In [6]:
import numpy as np
from spirit import simulation, state,quantities, hamiltonian,parameters,geometry,configuration,system,io
import numpy as np
import os
import matplotlib.pyplot as plt
import multiprocessing as mp
import plotly.graph_objects as go
import plotly.express as px
import pandas as pd
from scipy.stats import linregress
from tqdm import tqdm
import time
from datetime import timedelta

# df_all = pd.concat(results, ignore_index=True)

n_cycles = 88*2
# H_relax = 1.2
H_relax = 1.8
# H_relax_steps = 200
H_relax_steps = 100
dim = 10
concentration = 20
# gamma = 0.000001
gammas = [0.0002]*n_cycles
# /Users/jiakai/Desktop/SURF/code/_spirit/ln_MCS_decay_gammas_dim_4_with_SEM_errorbars_Ht2.3.html
df_all = pd.read_csv(f"/Users/jiakai/Desktop/SURF/code/_spirit/MCS_decay_gammas_dim_{dim}_with_SEM_errorbars.csv")

grouped = df_all.groupby(['gamma', 'i']).agg(
    chi_mean=('chi', 'mean'),
    chi_std=('chi', 'std')
).reset_index()

grouped['chi_sem'] = grouped['chi_std'] / (n_cycles ** 0.5)

fig = go.Figure()

unique_gammas = grouped['gamma'].unique()


fig_lnln = go.Figure()

for gamma in unique_gammas:
    df_gamma = grouped[grouped['gamma'] == gamma].copy()

    # Filter out invalid values
    df_gamma = df_gamma[(df_gamma['chi_mean'] > 0) & (np.log(df_gamma['chi_mean']) < 0)]

    # Compute safely: ln(i), ln(-ln(chi))
    df_gamma = df_gamma[(df_gamma['chi_mean'] > 0)]
    df_gamma['ln_i'] = np.log(df_gamma['i'])

    # Filter out cases where ln(chi) >= 0 → invalid for ln(-ln(chi))
    df_gamma = df_gamma[np.log(df_gamma['chi_mean']) < 0]
    df_gamma['ln_ln_chi'] = np.log(-np.log(df_gamma['chi_mean']))

    # Error propagation for ln(-ln(chi))
    df_gamma['ln_ln_chi_sem'] = (
        np.abs(1 / (np.log(df_gamma['chi_mean']) * df_gamma['chi_mean'])) * df_gamma['chi_sem']
    )

    # Scatter trace (data with error bars)
    fig_lnln.add_trace(go.Scatter(
        x=df_gamma['ln_i'],
        y=df_gamma['ln_ln_chi'],
        mode='lines+markers',
        name=f"γ = {gamma:.1e}",
        error_y=dict(
            type='data',
            array=df_gamma['ln_ln_chi_sem'],
            visible=True
        )
    ))

    # Linear fit
    x = df_gamma['ln_i']
    y = df_gamma['ln_ln_chi']
    if len(x) > 1:
        slope, intercept = np.polyfit(x, y, deg=1)
        x_fit = np.linspace(x.min(), x.max(), 100)
        y_fit = slope * x_fit + intercept

        fig_lnln.add_trace(go.Scatter(
            x=x_fit,
            y=y_fit,
            mode='lines',
            line=dict(dash='dot'),
            name=f"Fit γ={gamma:.1e}, slope={slope:.2f}"
        ))

    fig.show()


 ** On entry to DLASCL parameter number  4 had an illegal value
 ** On entry to DLASCL parameter number  4 had an illegal value
 ** On entry to DLASCL parameter number  4 had an illegal value
 ** On entry to DLASCL parameter number  4 had an illegal value
 ** On entry to DLASCL parameter number  5 had an illegal value
 ** On entry to DLASCL parameter number  4 had an illegal value


/Users/jiakai/Desktop/SURF/code/venv/lib/python3.13/site-packages/pandas/core/arraylike.py:399: RuntimeWarning:

divide by zero encountered in log

/Users/jiakai/Desktop/SURF/code/venv/lib/python3.13/site-packages/numpy/lib/_polynomial_impl.py:674: RuntimeWarning:

invalid value encountered in divide



LinAlgError: SVD did not converge in Linear Least Squares